# DuckDB × S3

DuckDB は `httpfs` 拡張を使うことで、**S3上のParquetファイルをそのまま直接クエリ**できます。  
ファイルをローカルにダウンロードする必要はありません。

## 目次
1. 準備（httpfs拡張・S3認証設定）
2. ローカルParquetをS3にアップロード
3. S3上のParquetを直接クエリ
4. ワイルドカードで複数ファイルをクエリ
5. ローカル vs S3 のクエリ速度比較

---
## 1. 準備

DuckDB でS3にアクセスするには `httpfs` 拡張が必要です。  
認証情報は `.env` から環境変数経由で読み込み、コードに直接書かないようにします。

In [ ]:
import duckdb
import os
from dotenv import load_dotenv

# .env から認証情報を読み込む
load_dotenv(dotenv_path="../../.env")

AWS_ACCESS_KEY_ID     = os.environ["DUCKDB_AWS_ACCESS_KEY_ID"]
AWS_SECRET_ACCESS_KEY = os.environ["DUCKDB_AWS_SECRET_ACCESS_KEY"]
S3_BUCKET             = os.environ["DUCKDB_S3_BUCKET"]
S3_REGION             = os.environ["DUCKDB_S3_REGION"]

PARQUET_DIR = "../data/parquet"
S3_PREFIX   = f"s3://{S3_BUCKET}/parquet"

con = duckdb.connect()

# httpfs 拡張をインストール・ロード
con.execute("INSTALL httpfs")
con.execute("LOAD httpfs")

# S3認証設定（環境変数から読み込んだ値を使用）
con.execute(f"""
    SET s3_region             = '{S3_REGION}';
    SET s3_access_key_id      = '{AWS_ACCESS_KEY_ID}';
    SET s3_secret_access_key  = '{AWS_SECRET_ACCESS_KEY}';
""")

print(f"DuckDB version : {duckdb.__version__}")
print(f"S3 bucket      : {S3_BUCKET}")
print(f"S3 region      : {S3_REGION}")
print("httpfs 拡張ロード完了")

---
## 2. ローカルParquetをS3にアップロード

DuckDB の `COPY TO 's3://...'` で直接S3に書き出せます。  
boto3などのAWSライブラリは不要です。

In [ ]:
# ローカルParquet → S3 にアップロード（3ファイル）
for table in ["customers", "products", "sales"]:
    local_path = f"{PARQUET_DIR}/{table}.parquet"
    s3_path    = f"{S3_PREFIX}/{table}.parquet"

    con.execute(f"""
        COPY (
            SELECT * FROM read_parquet('{local_path}')
        )
        TO '{s3_path}'
        (FORMAT PARQUET)
    """)
    print(f"✓ アップロード完了: {s3_path}")

In [ ]:
# 月別Parquetもアップロード
MONTHLY_DIR       = f"{PARQUET_DIR}/monthly"
S3_MONTHLY_PREFIX = f"{S3_PREFIX}/monthly"

for fname in sorted(os.listdir(MONTHLY_DIR)):
    if not fname.endswith(".parquet"):
        continue
    local_path = f"{MONTHLY_DIR}/{fname}"
    s3_path    = f"{S3_MONTHLY_PREFIX}/{fname}"

    con.execute(f"""
        COPY (
            SELECT * FROM read_parquet('{local_path}')
        )
        TO '{s3_path}'
        (FORMAT PARQUET)
    """)
    print(f"✓ {fname}")

---
## 3. S3上のParquetを直接クエリ

`read_parquet()` のパスを `s3://` に変えるだけです。  
ローカルと全く同じ構文で使えます。

In [ ]:
# S3上のsales.parquetを直接クエリ（ダウンロード不要）
rows = con.execute(f"""
    SELECT *
    FROM read_parquet('{S3_PREFIX}/sales.parquet')
    LIMIT 5
""").fetchall()

print(f"=== S3: sales.parquet 先頭5件 ===")
for row in rows:
    print(row)

In [ ]:
# S3上でJOINクエリ（3ファイルをまたぐ）
rows = con.execute(f"""
    SELECT
        c.name        AS 顧客名,
        p.product_name AS 商品名,
        s.amount      AS 金額
    FROM read_parquet('{S3_PREFIX}/sales.parquet')     s
    JOIN read_parquet('{S3_PREFIX}/customers.parquet') c ON s.customer_id = c.customer_id
    JOIN read_parquet('{S3_PREFIX}/products.parquet')  p ON s.product_id  = p.product_id
    ORDER BY s.amount DESC
    LIMIT 5
""").fetchall()

print(f"{'顧客名':<12} {'商品名':<20} {'金額':>10}")
print("-" * 45)
for row in rows:
    print(f"{row[0]:<12} {row[1]:<20} {row[2]:>10,.0f}")

---
## 4. ワイルドカードで複数ファイルをクエリ

S3上でもワイルドカード `*.parquet` が使えます。  
月別ファイルを1クエリでまとめて集計します。

In [ ]:
# S3上の月別Parquetをワイルドカードでまとめてクエリ
rows = con.execute(f"""
    SELECT
        strftime(sale_date, '%Y-%m') AS 年月,
        COUNT(*)                     AS 件数,
        SUM(amount)                  AS 売上合計
    FROM read_parquet('{S3_MONTHLY_PREFIX}/*.parquet')
    GROUP BY 年月
    ORDER BY 年月
""").fetchall()

print(f"{'年月':<10} {'件数':>6} {'売上合計':>12}")
print("-" * 32)
for row in rows:
    print(f"{row[0]:<10} {row[1]:>6} {row[2]:>12,.0f}")

print(f"\n→ S3上の {len(rows)} ファイルを1クエリで集計")

---
## 5. ローカル vs S3 のクエリ速度比較

同じSQLをローカルとS3で実行し、速度差を比較します。  
差が出る要因は**ネットワーク転送（ただし列指向なので転送量は最小限）**です。

In [ ]:
import time

QUERY_TEMPLATE = """
    SELECT
        strftime(sale_date, '%Y-%m') AS 年月,
        COUNT(*)                     AS 件数,
        SUM(amount)                  AS 売上合計
    FROM read_parquet('{path}')
    GROUP BY 年月
    ORDER BY 年月
"""

targets = [
    ("ローカル", f"{PARQUET_DIR}/sales.parquet"),
    ("S3      ", f"{S3_PREFIX}/sales.parquet"),
]

print(f"{'場所':<10} {'実行時間(秒)':>12}")
print("-" * 25)

for label, path in targets:
    start = time.time()
    con.execute(QUERY_TEMPLATE.format(path=path)).fetchall()
    elapsed = time.time() - start
    print(f"{label:<10} {elapsed:>12.3f} 秒")

---
## まとめ

| 機能 | 構文 |
|------|------|
| S3認証設定 | `SET s3_access_key_id / s3_secret_access_key` |
| S3からクエリ | `read_parquet('s3://bucket/file.parquet')` |
| S3へ書き出し | `COPY (...) TO 's3://bucket/file.parquet'` |
| S3ワイルドカード | `read_parquet('s3://bucket/dir/*.parquet')` |

**ポイント**  
- ローカルとS3で構文が全く同じ → 開発はローカル、本番はS3と切り替えやすい  
- 列指向なので必要な列のデータしか転送されない → ネットワークコストを抑えられる  
- S3上の複数ファイルをJOINすることも可能